# MMA Fight Phase + Pressure Classifier — Option C Pipeline

Multi-task K-fold CV comparing Conv3D CNN and LSTM on ~5-second MMA clips.
Uses **deterministic fighter identity** via YOLO preprocessing.

## Pipeline
1. **Section 2**: Mount Drive, configure paths
2. **Section 3**: Preprocessing — YOLO detects fighters, matches shorts color, creates identity masks
3. **Section 4-8**: Define dataset, models, training loop
4. **Section 9-10**: Train CNN and LSTM with K-fold CV
5. **Section 11-12**: Results and comparison

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Data layout on Drive:**
```
My Drive/mma_data/
  FightName/
    FightName_labels.csv    ← must have f1_color, f2_color columns
    clip_0002.mp4
    clip_0003.mp4
    ...
```

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics opencv-python-headless face_recognition torchreid gdown tensorboard

In [ ]:
import cv2, time, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torchvision import transforms
from collections import Counter
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

## 2 — Configuration

Edit `DATA_DIR` to point at your Drive folder. The `f1_color` / `f2_color` columns
in your CSV map the left/right broadcast name to a shorts color.

Supported colors: `red, blue, black, white, green, gold, gray, orange, purple`

In [ ]:
class cfg:
    DATA_DIR   = Path('/content/drive/MyDrive/mma_data')
    OUTPUT_DIR = Path('/content/outputs')

    PHASE_LABELS = ['Striking','Grappling/Ground Work','Clinch',
                    'Transition/Takedown','Neutral/Measuring Distance']
    NUM_PHASE_CLASSES = len(PHASE_LABELS)
    PHASE2IDX = {l:i for i,l in enumerate(PHASE_LABELS)}
    IDX2PHASE = {i:l for l,i in PHASE2IDX.items()}

    PRESSURE_LABELS = ['Fighter 1','Fighter 2','Mutual']
    NUM_PRESSURE_CLASSES = len(PRESSURE_LABELS)
    PRESSURE2IDX = {l:i for i,l in enumerate(PRESSURE_LABELS)}
    IDX2PRESSURE = {i:l for l,i in PRESSURE2IDX.items()}

    PHASE_LOSS_WEIGHT = 1.0; PRESSURE_LOSS_WEIGHT = 0.5
    NUM_FRAMES = 16; FRAME_HEIGHT = 112; FRAME_WIDTH = 112
    NUM_CHANNELS = 4  # RGB + identity mask
    BATCH_SIZE = 8; NUM_EPOCHS = 30; LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4; PATIENCE = 7; RANDOM_SEED = 42
    NUM_WORKERS = 2; K_FOLDS = 5; WARMUP_EPOCHS = 3
    TEMPORAL_JITTER = 0.15; MIXUP_ALPHA = 0.2
    LSTM_HIDDEN = 256; LSTM_LAYERS = 2; LSTM_DROPOUT = 0.3
    CNN_BASE_FILTERS = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(cfg.RANDOM_SEED)
print(f'Device: {cfg.DEVICE}')

## 3 — Preprocessing: Face → ReID → Color → Identity Masks

3-tier fighter identification per detected person:
1. **Face** — `face_recognition` library (when facing camera)
2. **ReID** — OSNet person re-identification (works from **any angle** — tattoos, hair, build, posture)
3. **Color** — shorts color HSV matching (fallback)

**Folder setup:**
```
FightName/
  FightName_labels.csv   ← must have f1_color, f2_color columns
  f1_face.jpg/png        ← reference image of Fighter 1 (upper body is fine)
  f2_face.jpg/png        ← reference image of Fighter 2
  clip_0002.mp4
```

The **same reference image** is used for both face encoding AND ReID body embedding.
So an upper-body shot like yours works perfectly — the face library extracts the face,
and OSNet encodes the full body appearance (tattoos, braids, skin tone, build).

In [ ]:
# ── HSV color ranges ──
COLOR_RANGES = {
    'red':   [((0,60,60),(12,255,255)), ((168,60,60),(180,255,255))],
    'blue':  [((100,60,50),(135,255,255))],
    'black': [((0,0,0),(180,100,70))],
    'white': [((0,0,180),(180,50,255))],
    'green': [((35,60,50),(85,255,255))],
    'gold':  [((15,60,50),(35,255,255))],
    'gray':  [((0,0,50),(180,50,180))],
    'orange':[((10,60,50),(20,255,255))],
    'purple':[((130,60,50),(165,255,255))],
}
FACE_MATCH_THRESHOLD = 0.6
REID_MATCH_THRESHOLD = 0.70
REID_MARGIN_THRESHOLD = 0.05

REID_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── YOLO ──
def detect_persons(yolo_model, frame, min_conf=0.35):
    results = yolo_model(frame, classes=[0], verbose=False, conf=min_conf)
    dets = []
    for box in results[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
        dets.append((x1,y1,x2,y2,box.conf[0].item(),(x2-x1)*(y2-y1)))
    dets.sort(key=lambda d: d[5], reverse=True)
    return dets[:2]

# ── Tier 1: Face ──
import face_recognition as face_rec

def encode_ref_face(image_path):
    img = face_rec.load_image_file(str(image_path))
    locs = face_rec.face_locations(img, model='hog')
    if not locs: print(f'    No face in {Path(image_path).name}'); return None
    areas = [(b-t)*(r-l) for t,r,b,l in locs]
    encs = face_rec.face_encodings(img, [locs[int(np.argmax(areas))]])
    return encs[0] if encs else None

def find_face_in_bbox(frame_rgb, x1,y1,x2,y2):
    h = y2-y1
    crop = frame_rgb[max(0,y1-int(h*.05)):min(frame_rgb.shape[0],y1+int(h*.45)),
                     max(0,x1-10):min(frame_rgb.shape[1],x2+10)]
    if crop.size==0: return None
    locs = face_rec.face_locations(crop, model='hog')
    if not locs: return None
    encs = face_rec.face_encodings(crop, locs[:1])
    return encs[0] if encs else None

def match_face_to_refs(enc, ref_f1, ref_f2):
    dists = []
    if ref_f1 is not None: dists.append(('f1',face_rec.face_distance([ref_f1],enc)[0]))
    if ref_f2 is not None: dists.append(('f2',face_rec.face_distance([ref_f2],enc)[0]))
    if not dists: return None,None
    dists.sort(key=lambda x:x[1])
    fid,d = dists[0]
    return (fid,d) if d < FACE_MATCH_THRESHOLD else (None,None)

# ── Tier 2: ReID (OSNet) ──
import torchreid
from PIL import Image as PILImage

reid_model = torchreid.utils.FeatureExtractor(
    model_name='osnet_x0_25',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print(f'OSNet ReID loaded on {"cuda" if torch.cuda.is_available() else "cpu"}')

def extract_reid_embedding(frame_rgb, x1,y1,x2,y2):
    h,w = y2-y1, x2-x1
    crop = frame_rgb[max(0,y1+int(h*.02)):min(frame_rgb.shape[0],y2-int(h*.02)),
                     max(0,x1+int(w*.05)):min(frame_rgb.shape[1],x2-int(w*.05))]
    if crop.size==0: return None
    tensor = REID_TRANSFORM(PILImage.fromarray(crop)).unsqueeze(0)
    device = next(reid_model.model.parameters()).device
    with torch.no_grad():
        feat = reid_model.model(tensor.to(device))
    feat = torch.nn.functional.normalize(feat, p=2, dim=1)
    return feat.cpu().squeeze(0).numpy()

def encode_ref_reid(image_path):
    img = cv2.imread(str(image_path))
    if img is None: return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h,w = rgb.shape[:2]
    return extract_reid_embedding(rgb, 0, 0, w, h)

def match_reid(emb, ref_f1, ref_f2):
    scores = []
    if ref_f1 is not None: scores.append(('f1',float(np.dot(emb,ref_f1))))
    if ref_f2 is not None: scores.append(('f2',float(np.dot(emb,ref_f2))))
    if not scores: return None,None
    scores.sort(key=lambda x:x[1], reverse=True)
    fid,s = scores[0]
    if s < REID_MATCH_THRESHOLD: return None,None
    if len(scores)==2 and scores[0][1]-scores[1][1] < REID_MARGIN_THRESHOLD: return None,None
    return fid,s

# ── Tier 3: Color ──
def get_shorts_region(hsv, x1,y1,x2,y2):
    h,w = y2-y1,x2-x1
    return hsv[max(0,y1+int(h*.35)):min(hsv.shape[0],y1+int(h*.60)),
              max(0,x1+int(w*.15)):min(hsv.shape[1],x2-int(w*.15))]

def classify_shorts_color(hsv_crop):
    if hsv_crop.size==0: return 'unknown',0.0
    total = hsv_crop.shape[0]*hsv_crop.shape[1]
    best,best_r = 'unknown',0.0
    for name,ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv_crop.shape[:2],dtype=np.uint8)
        for lo,hi in ranges: mask |= cv2.inRange(hsv_crop,np.array(lo),np.array(hi))
        r = mask.sum()/255/total
        if r > best_r: best_r,best = r,name
    return best,best_r

def assign_by_color(dets, hsv, f1c, f2c):
    if len(dets)<2:
        if len(dets)==1:
            x1,y1,x2,y2,_,_ = dets[0]
            c,_ = classify_shorts_color(get_shorts_region(hsv,x1,y1,x2,y2))
            if c==f1c: return (x1,y1,x2,y2),None
            elif c==f2c: return None,(x1,y1,x2,y2)
        return None,None
    res = []
    for d in dets[:2]:
        x1,y1,x2,y2,_,_ = d
        c,r = classify_shorts_color(get_shorts_region(hsv,x1,y1,x2,y2))
        res.append(((x1,y1,x2,y2),c,r))
    ba,ca = res[0][0],res[0][1]; bb,cb = res[1][0],res[1][1]
    if ca==f1c and cb==f2c: return ba,bb
    elif ca==f2c and cb==f1c: return bb,ba
    elif ca==f1c: return ba,bb
    elif cb==f1c: return bb,ba
    elif ca==f2c: return bb,ba
    elif cb==f2c: return ba,bb
    else: return (ba,bb) if ba[0]<bb[0] else (bb,ba)

# ── 3-Tier hybrid: Face → ReID → Color ──
def assign_fighter_identity(dets, frame_rgb, frame_bgr, hsv, f1c, f2c,
                            ref_f1=None, ref_f2=None,
                            ref_f1_reid=None, ref_f2_reid=None):
    if len(dets)<1: return None,None,'none'
    def bbox(i):
        x1,y1,x2,y2,_,_ = dets[i]; return (x1,y1,x2,y2)
    def try_assign(assigns):
        if len(assigns)==2 and set(assigns.values())=={'f1','f2'}:
            bbs = {assigns[i]:bbox(i) for i in assigns}
            return bbs.get('f1'),bbs.get('f2'),True
        elif len(assigns)==1:
            idx,fid = list(assigns.items())[0]
            oi = 1-idx if len(dets)>=2 else None
            ob = bbox(oi) if oi is not None and oi<len(dets) else None
            return (bbox(idx),ob,True) if fid=='f1' else (ob,bbox(idx),True)
        return None,None,False
    # Tier 1: Face
    if ref_f1 is not None or ref_f2 is not None:
        a = {}
        for i in range(min(2,len(dets))):
            x1,y1,x2,y2,_,_ = dets[i]
            enc = find_face_in_bbox(frame_rgb,x1,y1,x2,y2)
            if enc is not None:
                fid,_ = match_face_to_refs(enc,ref_f1,ref_f2)
                if fid: a[i]=fid
        f1b,f2b,ok = try_assign(a)
        if ok: return f1b,f2b,'face' if len(a)==2 else 'face_partial'
    # Tier 2: ReID
    if ref_f1_reid is not None or ref_f2_reid is not None:
        a = {}
        for i in range(min(2,len(dets))):
            x1,y1,x2,y2,_,_ = dets[i]
            emb = extract_reid_embedding(frame_rgb,x1,y1,x2,y2)
            if emb is not None:
                fid,_ = match_reid(emb,ref_f1_reid,ref_f2_reid)
                if fid: a[i]=fid
        f1b,f2b,ok = try_assign(a)
        if ok: return f1b,f2b,'reid' if len(a)==2 else 'reid_partial'
    # Tier 3: Color
    f1b,f2b = assign_by_color(dets,hsv,f1c,f2c)
    return f1b,f2b,'color' if (f1b or f2b) else 'none'

def create_identity_mask(shape, f1b, f2b, sz=(112,112)):
    h,w = shape[:2]; mask = np.zeros((h,w),dtype=np.float32)
    if f1b: mask[f1b[1]:f1b[3],f1b[0]:f1b[2]] = 1.0
    if f2b: mask[f2b[1]:f2b[3],f2b[0]:f2b[2]] = -1.0
    if f1b and f2b:
        ox1,oy1 = max(f1b[0],f2b[0]),max(f1b[1],f2b[1])
        ox2,oy2 = min(f1b[2],f2b[2]),min(f1b[3],f2b[3])
        if ox1<ox2 and oy1<oy2: mask[oy1:oy2,ox1:ox2] = 0.0
    return cv2.resize(mask,(sz[1],sz[0]),interpolation=cv2.INTER_NEAREST)

print('3-tier preprocessing ready: Face → ReID (OSNet) → Color')

In [ ]:
# ── Run preprocessing: Face → ReID → Color ──
yolo_model = YOLO('yolov8n.pt')
print('YOLOv8 loaded\n')

data_dir = cfg.DATA_DIR
for fight_dir in sorted(data_dir.iterdir()):
    if not fight_dir.is_dir(): continue
    csvs = list(fight_dir.glob('*_labels.csv'))
    if not csvs: continue
    df = pd.read_csv(csvs[0])
    if 'f1_color' not in df.columns:
        print(f'{fight_dir.name}: missing f1_color/f2_color'); continue
    f1c = df['f1_color'].iloc[0].strip().lower()
    f2c = df['f2_color'].iloc[0].strip().lower()
    print(f'\n{"─"*50}')
    print(f'{fight_dir.name}  |  F1={f1c}  F2={f2c}')

    # Load references — same image for face AND ReID
    ref_f1_face, ref_f2_face = None, None
    ref_f1_reid, ref_f2_reid = None, None
    for label, prefix in [('F1','f1_face'),('F2','f2_face')]:
        rp = None
        for ext in ['jpg','jpeg','png']:
            p = fight_dir / f'{prefix}.{ext}'
            if p.exists(): rp = p; break
        if rp is None: print(f'  {label} ref: not found'); continue
        # Face
        try:
            enc = encode_ref_face(rp)
            if enc is not None:
                if label=='F1': ref_f1_face = enc
                else: ref_f2_face = enc
                print(f'  {label} face: encoded')
            else: print(f'  {label} face: no face detected')
        except Exception as e: print(f'  {label} face error: {e}')
        # ReID
        emb = encode_ref_reid(rp)
        if emb is not None:
            if label=='F1': ref_f1_reid = emb
            else: ref_f2_reid = emb
            print(f'  {label} ReID: encoded (512-d)')

    tiers = []
    if ref_f1_face or ref_f2_face: tiers.append('face')
    if ref_f1_reid or ref_f2_reid: tiers.append('ReID')
    tiers.append('color')
    print(f'  Active: {" → ".join(tiers)}')

    if df['excluded'].dtype == object:
        df['excluded'] = df['excluded'].str.strip().str.lower() == 'true'
    valid = df[~df['excluded']].dropna(subset=['phase_label'])
    total_stats = Counter()
    for _, row in valid.iterrows():
        cp = fight_dir / row['saved_filename']
        mp = fight_dir / row['saved_filename'].replace('.mp4','_identity.npy')
        if not cp.exists() or mp.exists(): continue
        cap = cv2.VideoCapture(str(cp))
        masks = []; fstats = Counter()
        while True:
            ret, frame = cap.read()
            if not ret: break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            dets = detect_persons(yolo_model, frame)
            f1b,f2b,method = assign_fighter_identity(
                dets, rgb, frame, hsv, f1c, f2c,
                ref_f1_face, ref_f2_face, ref_f1_reid, ref_f2_reid)
            masks.append(create_identity_mask(frame.shape, f1b, f2b))
            fstats[method] += 1
        cap.release()
        if masks:
            np.save(mp, np.stack(masks))
            total_stats.update(fstats)
            n = sum(fstats.values())
            parts = [f'{m}={fstats[m]/n*100:.0f}%' for m in ['face','face_partial','reid','reid_partial','color','none'] if fstats.get(m,0)>0]
            print(f'  {row["saved_filename"]}: {" ".join(parts)}')
    total = sum(total_stats.values())
    if total:
        print(f'\n  TOTAL ({total} frames):')
        for m in ['face','face_partial','reid','reid_partial','color','none']:
            c = total_stats.get(m,0)
            if c: print(f'    {m:14s} {c:>5d} ({c/total*100:.1f}%)')
print('\nPreprocessing complete')

### Verify preprocessing — visual debug

This cell picks sample clips and shows three rows per clip:
- **Row 1**: YOLO bounding boxes with assigned fighter labels + shorts crop regions
- **Row 2**: Identity mask (blue=F1, red=F2)
- **Row 3**: Mask overlaid on the frame — you should see blue tint on F1, red tint on F2

Check for: wrong assignments (F1/F2 swapped), missed detections (NONE), referee detected as fighter.

In [ ]:
def debug_frame(yolo_model, frame, f1c, f2c,
                ref_f1_face=None, ref_f2_face=None,
                ref_f1_reid=None, ref_f2_reid=None):
    """Run full 3-tier pipeline on one frame, return annotated frame + mask + overlay."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    dets = detect_persons(yolo_model, frame)
    det_colors = []
    for d in dets:
        x1,y1,x2,y2,_,_ = d
        c,r = classify_shorts_color(get_shorts_region(hsv,x1,y1,x2,y2))
        det_colors.append((c,r))
    f1b, f2b, method = assign_fighter_identity(
        dets, rgb, frame, hsv, f1c, f2c,
        ref_f1=ref_f1_face, ref_f2=ref_f2_face,
        ref_f1_reid=ref_f1_reid, ref_f2_reid=ref_f2_reid)
    mask = create_identity_mask(frame.shape, f1b, f2b)
    # Annotate
    ann = frame.copy()
    for i,(d,(c,r)) in enumerate(zip(dets, det_colors)):
        x1,y1,x2,y2,_,_ = d
        cv2.rectangle(ann,(x1,y1),(x2,y2),(128,128,128),1)
        cv2.putText(ann,f'{c}({r:.0%})',(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.5,(128,128,128),1)
    if f1b:
        cv2.rectangle(ann,(f1b[0],f1b[1]),(f1b[2],f1b[3]),(255,100,0),3)
        cv2.putText(ann,f'F1 ({f1c})',(f1b[0],f1b[1]-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,100,0),2)
    if f2b:
        cv2.rectangle(ann,(f2b[0],f2b[1]),(f2b[2],f2b[3]),(0,0,255),3)
        cv2.putText(ann,f'F2 ({f2c})',(f2b[0],f2b[1]-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
    # Overlay
    ov = frame.copy().astype(np.float32)
    mf = cv2.resize(mask,(frame.shape[1],frame.shape[0]),interpolation=cv2.INTER_NEAREST)
    f1r = mf > 0.5; f2r = mf < -0.5
    ov[f1r,0]=ov[f1r,0]*.5+255*.5; ov[f1r,1]*=.5; ov[f1r,2]*=.5
    ov[f2r,0]*=.5; ov[f2r,1]*=.5; ov[f2r,2]=ov[f2r,2]*.5+255*.5
    status = f'{method}' if (f1b or f2b) else 'NONE'
    return ann, mask, ov.astype(np.uint8), status

# ── Run debug on sample clips ──
data_dir = cfg.DATA_DIR
for fight_dir in sorted(data_dir.iterdir()):
    if not fight_dir.is_dir(): continue
    csvs = list(fight_dir.glob('*_labels.csv'))
    if not csvs: continue
    df = pd.read_csv(csvs[0])
    if 'f1_color' not in df.columns: continue
    f1c = df['f1_color'].iloc[0].strip().lower()
    f2c = df['f2_color'].iloc[0].strip().lower()
    if df['excluded'].dtype == object:
        df['excluded'] = df['excluded'].str.strip().str.lower() == 'true'
    valid = df[~df['excluded']].dropna(subset=['phase_label'])

    # Load reference faces/bodies for this fight (reuse from preprocessing)
    d_ref_f1_face, d_ref_f2_face = None, None
    d_ref_f1_reid, d_ref_f2_reid = None, None
    for label, prefix in [('F1','f1_face'),('F2','f2_face')]:
        rp = None
        for ext in ['jpg','jpeg','png']:
            p = fight_dir / f'{prefix}.{ext}'
            if p.exists(): rp = p; break
        if rp is None: continue
        ref_bgr = cv2.imread(str(rp))
        # Face encoding
        try:
            enc = encode_ref_face(rp)
            if enc is not None:
                if label=='F1': d_ref_f1_face = enc
                else: d_ref_f2_face = enc
        except: pass
        # Body descriptor
        h,w = ref_bgr.shape[:2]
        bd = encode_ref_reid(rp) if 'encode_ref_reid' in dir() else None
        if bd is not None:
            if label=='F1': d_ref_f1_reid = bd
            else: d_ref_f2_reid = bd

    # Pick 3 clips: start, middle, end of fight
    sample_idx = np.linspace(0, len(valid)-1, min(3, len(valid)), dtype=int)
    for si in sample_idx:
        row = valid.iloc[si]
        cp = fight_dir / row['saved_filename']
        if not cp.exists(): continue
        cap = cv2.VideoCapture(str(cp))
        frames = []
        while True:
            ret, fr = cap.read()
            if not ret: break
            frames.append(fr)
        cap.release()
        if not frames: continue
        fidx = np.linspace(0, len(frames)-1, 6, dtype=int)
        fig, axes = plt.subplots(3, 6, figsize=(24, 10))
        for c, fi in enumerate(fidx):
            ann, mask, ov, status = debug_frame(
                yolo_model, frames[fi], f1c, f2c,
                d_ref_f1_face, d_ref_f2_face,
                d_ref_f1_reid, d_ref_f2_reid)
            axes[0,c].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)); axes[0,c].set_title(f'Frame {fi}',fontsize=9); axes[0,c].axis('off')
            axes[1,c].imshow(mask, cmap='RdBu', vmin=-1, vmax=1); axes[1,c].set_title('Mask',fontsize=9); axes[1,c].axis('off')
            axes[2,c].imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)); axes[2,c].set_title(status,fontsize=9); axes[2,c].axis('off')
        axes[0,0].set_ylabel('Detections',fontsize=11)
        axes[1,0].set_ylabel('Mask',fontsize=11)
        axes[2,0].set_ylabel('Overlay',fontsize=11)
        plt.suptitle(f'{fight_dir.name} / {row["saved_filename"]}  |  F1={f1c}  F2={f2c}', fontsize=12, fontweight='bold')
        plt.tight_layout(); plt.show()
        print()

print('Each frame shows which method was used: face / body / color / none')
print('Blue tint = F1, Red tint = F2. If swapped, fix f1_color/f2_color in CSV.')

## 4 — Dataset

In [ ]:
# ── Transforms ──
train_spatial_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop((cfg.FRAME_HEIGHT,cfg.FRAME_WIDTH), scale=(0.75,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.3,hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_spatial_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((cfg.FRAME_HEIGHT,cfg.FRAME_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def discover_clips(data_dir):
    all_rows = []
    for fight_dir in sorted(Path(data_dir).iterdir()):
        if not fight_dir.is_dir(): continue
        csvs = list(fight_dir.glob('*_labels.csv'))
        if not csvs: continue
        df = pd.read_csv(csvs[0])
        if df['excluded'].dtype == object:
            df['excluded'] = df['excluded'].str.strip().str.lower() == 'true'
        df = df[~df['excluded']].dropna(subset=['phase_label','pressure_label']).copy()
        df['clip_path'] = df['saved_filename'].apply(lambda f: str(fight_dir / f))
        df['fight'] = fight_dir.name
        all_rows.append(df)
    merged = pd.concat(all_rows, ignore_index=True)
    print(f'Discovered {len(merged)} valid clips across {len(all_rows)} fight(s)')
    return merged

class VideoDataset(Dataset):
    def __init__(self, records, transform=None, temporal_jitter=0.0):
        self.records = records.reset_index(drop=True)
        self.transform = transform or val_spatial_transform
        self.temporal_jitter = temporal_jitter
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        phase = cfg.PHASE2IDX[row['phase_label']]
        pressure = cfg.PRESSURE2IDX[row['pressure_label']]
        frames = self._load_frames(row['clip_path'])
        id_masks = self._load_masks(row['clip_path'], len(frames))
        indices = self._sample_idx(len(frames))
        frames = [frames[i] for i in indices]
        id_masks = id_masks[indices]
        rgb = torch.stack([self.transform(f) for f in frames], dim=1)  # (3,T,H,W)
        m = []
        for mk in id_masks:
            if mk.shape[0]!=cfg.FRAME_HEIGHT or mk.shape[1]!=cfg.FRAME_WIDTH:
                mk = cv2.resize(mk,(cfg.FRAME_WIDTH,cfg.FRAME_HEIGHT),interpolation=cv2.INTER_NEAREST)
            m.append(torch.from_numpy(mk).float())
        mask_ch = torch.stack(m, dim=0).unsqueeze(0)  # (1,T,H,W)
        return torch.cat([rgb, mask_ch], dim=0), phase, pressure  # (4,T,H,W)
    def _load_frames(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if not frames: raise RuntimeError(f'No frames from {path}')
        return frames
    def _load_masks(self, video_path, n):
        mp = video_path.replace('.mp4','_identity.npy')
        try: return np.load(mp)
        except FileNotFoundError:
            return np.zeros((n,cfg.FRAME_HEIGHT,cfg.FRAME_WIDTH), dtype=np.float32)
    def _sample_idx(self, n):
        if n < cfg.NUM_FRAMES:
            return list(range(n)) + [n-1]*(cfg.NUM_FRAMES-n)
        ideal = np.linspace(0,n-1,cfg.NUM_FRAMES)
        if self.temporal_jitter > 0:
            gap = (n-1)/max(cfg.NUM_FRAMES-1,1)
            ideal += np.random.uniform(-self.temporal_jitter*gap, self.temporal_jitter*gap, cfg.NUM_FRAMES)
        return np.clip(ideal.round().astype(int), 0, n-1)

def mixup_batch(videos, ph, pr, alpha=cfg.MIXUP_ALPHA):
    if alpha <= 0: return videos,(ph,ph,1.0),(pr,pr,1.0)
    lam = max(np.random.beta(alpha,alpha), 0.5)
    perm = torch.randperm(videos.size(0))
    return lam*videos+(1-lam)*videos[perm], (ph,ph[perm],lam), (pr,pr[perm],lam)

def compute_class_weights(labels, nc):
    counts = Counter(labels); n = len(labels)
    w = torch.zeros(nc)
    for c in range(nc): w[c] = n/(nc*counts.get(c,1))
    return w

print('Dataset utilities ready')

In [ ]:
records = discover_clips(cfg.DATA_DIR)
print('\nPhase distribution:')
for l,i in cfg.PHASE2IDX.items(): print(f'  [{i}] {l:35s} {(records["phase_label"]==l).sum():>4d}')
print('\nPressure distribution:')
for l,i in cfg.PRESSURE2IDX.items(): print(f'  [{i}] {l:15s} {(records["pressure_label"]==l).sum():>4d}')
ds = VideoDataset(records.head(1), transform=val_spatial_transform)
v,ph,pr = ds[0]
print(f'\nSample: {v.shape}  (4 channels = RGB + identity mask)')
print(f'Identity channel range: [{v[3].min():.1f}, {v[3].max():.1f}]')

## 5 — Models

Both accept **(4, 16, 112, 112)** — 3 RGB channels + 1 identity mask channel.
Both output **(phase_logits, pressure_logits)** via a shared-backbone, dual-head architecture.

In [ ]:
def _init_weights(m):
    if isinstance(m,(nn.Conv2d,nn.Conv3d)):
        nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m,nn.Linear):
        nn.init.kaiming_normal_(m.weight,mode='fan_in',nonlinearity='relu')
        nn.init.zeros_(m.bias)
    elif isinstance(m,(nn.BatchNorm2d,nn.BatchNorm3d)):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    elif isinstance(m,nn.LSTM):
        for n,p in m.named_parameters():
            if 'weight_ih' in n: nn.init.kaiming_normal_(p,mode='fan_in',nonlinearity='relu')
            elif 'weight_hh' in n: nn.init.orthogonal_(p)
            elif 'bias' in n:
                nn.init.zeros_(p); hs=p.shape[0]//4; p.data[hs:2*hs].fill_(1.0)

class _DualHead(nn.Module):
    def __init__(self, inf, do=0.4):
        super().__init__()
        self.dropout = nn.Dropout(do)
        self.head_phase = nn.Linear(inf, cfg.NUM_PHASE_CLASSES)
        self.head_pressure = nn.Linear(inf, cfg.NUM_PRESSURE_CLASSES)
    def forward(self, x):
        x = self.dropout(x)
        return self.head_phase(x), self.head_pressure(x)

class Conv3DClassifier(nn.Module):
    def __init__(self, f=cfg.CNN_BASE_FILTERS, ic=cfg.NUM_CHANNELS):
        super().__init__()
        def blk(i,o,pk=(2,2,2)):
            return nn.Sequential(nn.Conv3d(i,o,3,padding=1),nn.BatchNorm3d(o),nn.ReLU(True),nn.MaxPool3d(pk))
        self.features = nn.Sequential(blk(ic,f,(1,2,2)),blk(f,f*2),blk(f*2,f*4),blk(f*4,f*8))
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.heads = _DualHead(f*8,0.4)
        self.apply(_init_weights)
    def forward(self, x):
        return self.heads(self.pool(self.features(x)).flatten(1))

class _FrameEncoder(nn.Module):
    def __init__(self, od=256, ic=cfg.NUM_CHANNELS):
        super().__init__()
        def blk(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(ic,64),blk(64,128),blk(128,256),blk(256,od))
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x): return self.pool(self.features(x)).flatten(1)

class LSTMClassifier(nn.Module):
    def __init__(self, ed=256, hs=cfg.LSTM_HIDDEN, nl=cfg.LSTM_LAYERS, do=cfg.LSTM_DROPOUT):
        super().__init__()
        self.encoder = _FrameEncoder(ed)
        self.lstm = nn.LSTM(ed,hs,nl,batch_first=True,dropout=do if nl>1 else 0.)
        self.heads = _DualHead(hs,do)
        self.apply(_init_weights)
    def forward(self, x):
        B,C,T,H,W = x.shape
        feats = self.encoder(x.permute(0,2,1,3,4).contiguous().view(B*T,C,H,W)).view(B,T,-1)
        return self.heads(self.lstm(feats)[0][:,-1,:])

def build_model(name):
    m = Conv3DClassifier() if name=='cnn' else LSTMClassifier()
    return m.to(cfg.DEVICE)

dummy = torch.randn(2,4,cfg.NUM_FRAMES,cfg.FRAME_HEIGHT,cfg.FRAME_WIDTH).to(cfg.DEVICE)
for n in ['cnn','lstm']:
    m=build_model(n); p=sum(x.numel() for x in m.parameters() if x.requires_grad)
    ph,pr=m(dummy); print(f'{n.upper():5s} phase={ph.shape} pressure={pr.shape} params={p:,}')
del dummy; print('Models ready')

## 6 — Training loop

In [ ]:
class WarmupThenReduce:
    def __init__(self, opt, we, blr):
        self.opt,self.we,self.blr = opt,we,blr
        self.plat = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=3)
        self.ep = 0
    def step(self, vl=None):
        self.ep += 1
        if self.ep <= self.we:
            lr = self.blr*(self.ep/self.we)
            for pg in self.opt.param_groups: pg['lr'] = lr
        else: self.plat.step(vl)
    def get_lr(self): return self.opt.param_groups[0]['lr']

def train_model(model, tl, vl, name, pw=None, prw=None, fold=None):
    tag = f'{name}_fold{fold}' if fold is not None else name
    ckpt = cfg.OUTPUT_DIR / f'best_{tag}.pt'
    cpw = pw.to(cfg.DEVICE) if pw is not None else None
    cprw = prw.to(cfg.DEVICE) if prw is not None else None
    cph = nn.CrossEntropyLoss(weight=cpw); cpr = nn.CrossEntropyLoss(weight=cprw)
    opt = torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=cfg.LEARNING_RATE,weight_decay=cfg.WEIGHT_DECAY)
    sched = WarmupThenReduce(opt,cfg.WARMUP_EPOCHS,cfg.LEARNING_RATE)
    history = {'train_loss':[],'val_loss':[],'train_phase_acc':[],'val_phase_acc':[],'train_pressure_acc':[],'val_pressure_acc':[]}
    best_vl,pat = float('inf'),0; umx = cfg.MIXUP_ALPHA > 0
    for ep in range(1,cfg.NUM_EPOCHS+1):
        t0=time.time(); model.train(); rl,phc,prc,tot = 0.,0,0,0
        for v,ph,pr in tl:
            v,ph,pr = v.to(cfg.DEVICE),ph.to(cfg.DEVICE),pr.to(cfg.DEVICE)
            if umx:
                v,(pha,phb,lam),(pra,prb,_) = mixup_batch(v,ph,pr)
                v=v.to(cfg.DEVICE);pha,phb=pha.to(cfg.DEVICE),phb.to(cfg.DEVICE)
                pra,prb=pra.to(cfg.DEVICE),prb.to(cfg.DEVICE)
            opt.zero_grad()
            phl,prl = model(v)
            if umx:
                lph=lam*cph(phl,pha)+(1-lam)*cph(phl,phb)
                lpr=lam*cpr(prl,pra)+(1-lam)*cpr(prl,prb)
                phc+=(phl.argmax(1)==pha).sum().item();prc+=(prl.argmax(1)==pra).sum().item()
            else:
                lph=cph(phl,ph);lpr=cpr(prl,pr)
                phc+=(phl.argmax(1)==ph).sum().item();prc+=(prl.argmax(1)==pr).sum().item()
            loss=cfg.PHASE_LOSS_WEIGHT*lph+cfg.PRESSURE_LOSS_WEIGHT*lpr
            loss.backward();opt.step();rl+=loss.item()*v.size(0);tot+=v.size(0)
        tls=rl/tot;tpa=phc/tot;tra=prc/tot
        vloss,vpa,vra = _ev(model,vl,cph,cpr)
        sched.step(vloss)
        for k,val in [('train_loss',tls),('val_loss',vloss),('train_phase_acc',tpa),
                      ('val_phase_acc',vpa),('train_pressure_acc',tra),('val_pressure_acc',vra)]:
            history[k].append(val)
        print(f'  [{tag}] Ep {ep:>2}/{cfg.NUM_EPOCHS} loss={tls:.4f} ph={tpa:.3f}/{vpa:.3f} '
              f'pr={tra:.3f}/{vra:.3f} lr={sched.get_lr():.1e} ({time.time()-t0:.1f}s)')
        if vloss < best_vl: best_vl=vloss;pat=0;torch.save(model.state_dict(),ckpt)
        else:
            pat+=1
            if pat>=cfg.PATIENCE: print(f'  Early stopping at epoch {ep}'); break
    model.load_state_dict(torch.load(ckpt,map_location=cfg.DEVICE,weights_only=True))
    return history

@torch.no_grad()
def _ev(model,loader,cph,cpr):
    model.eval();rl,phc,prc,tot=0.,0,0,0
    for v,ph,pr in loader:
        v,ph,pr=v.to(cfg.DEVICE),ph.to(cfg.DEVICE),pr.to(cfg.DEVICE)
        phl,prl=model(v)
        loss=cfg.PHASE_LOSS_WEIGHT*cph(phl,ph)+cfg.PRESSURE_LOSS_WEIGHT*cpr(prl,pr)
        rl+=loss.item()*v.size(0);phc+=(phl.argmax(1)==ph).sum().item()
        prc+=(prl.argmax(1)==pr).sum().item();tot+=v.size(0)
    return rl/tot,phc/tot,prc/tot

@torch.no_grad()
def collect_preds(model,loader):
    model.eval();pht,php,prt,prp=[],[],[],[]
    for v,ph,pr in loader:
        v=v.to(cfg.DEVICE);phl,prl=model(v)
        php.extend(phl.argmax(1).cpu().numpy());pht.extend(ph.numpy())
        prp.extend(prl.argmax(1).cpu().numpy());prt.extend(pr.numpy())
    return np.array(pht),np.array(php),np.array(prt),np.array(prp)

print('Training utilities ready')

## 7 — K-Fold runner

In [ ]:
def build_kfold_loaders(data_dir=cfg.DATA_DIR):
    records = discover_clips(data_dir)
    phl = records['phase_label'].map(cfg.PHASE2IDX).values
    prl = records['pressure_label'].map(cfg.PRESSURE2IDX).values
    mn = min(Counter(phl).values())
    if mn >= cfg.K_FOLDS:
        kf=StratifiedKFold(cfg.K_FOLDS,shuffle=True,random_state=cfg.RANDOM_SEED)
        it=kf.split(records,phl); print(f'{cfg.K_FOLDS}-fold Stratified CV')
    else:
        kf=KFold(cfg.K_FOLDS,shuffle=True,random_state=cfg.RANDOM_SEED)
        it=kf.split(records); print(f'{cfg.K_FOLDS}-fold CV (min class={mn})')
    for fi,(tri,vi) in enumerate(it):
        tds=VideoDataset(records.iloc[tri],train_spatial_transform,cfg.TEMPORAL_JITTER)
        vds=VideoDataset(records.iloc[vi],val_spatial_transform,0.)
        tl=DataLoader(tds,cfg.BATCH_SIZE,shuffle=True,num_workers=cfg.NUM_WORKERS,pin_memory=True)
        vl=DataLoader(vds,cfg.BATCH_SIZE,shuffle=False,num_workers=cfg.NUM_WORKERS,pin_memory=True)
        pw=compute_class_weights(phl[tri],cfg.NUM_PHASE_CLASSES)
        prw=compute_class_weights(prl[tri],cfg.NUM_PRESSURE_CLASSES)
        print(f'  Fold {fi+1}/{cfg.K_FOLDS} train={len(tri)} val={len(vi)}')
        yield fi,tl,vl,pw,prw

def run_kfold(name):
    apht,aphp,aprt,aprp=[],[],[],[]
    fpa,fra=[],[]
    for fi,tl,vl,pw,prw in build_kfold_loaders():
        print(f'\n  {name.upper()} Fold {fi+1}/{cfg.K_FOLDS}')
        model=build_model(name)
        if fi==0: print(f'  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
        h=train_model(model,tl,vl,name,pw,prw,fi)
        pht,php,prt,prp=collect_preds(model,vl)
        apht.append(pht);aphp.append(php);aprt.append(prt);aprp.append(prp)
        fpa.append(max(h['val_phase_acc']));fra.append(max(h['val_pressure_acc']))
        print(f'  Best: phase={fpa[-1]:.3f} pressure={fra[-1]:.3f}')
    return (np.concatenate(apht),np.concatenate(aphp),
            np.concatenate(aprt),np.concatenate(aprp),fpa,fra)

print('K-Fold runner ready')

## 8 — Train CNN

In [ ]:
cnn_pht,cnn_php,cnn_prt,cnn_prp,cnn_fpa,cnn_fra = run_kfold('cnn')

## 9 — Train LSTM

In [ ]:
lstm_pht,lstm_php,lstm_prt,lstm_prp,lstm_fpa,lstm_fra = run_kfold('lstm')

## 10 — Results

In [ ]:
def show_results(name, yt, yp, fa, labels, task):
    ma=np.mean(fa);sa=np.std(fa)
    mf1=f1_score(yt,yp,average='macro',zero_division=0)
    print(f'\n{name.upper()} — {task.upper()}')
    print(f'Fold accs: {[f"{a:.3f}" for a in fa]}  Mean: {ma:.3f}±{sa:.3f}  F1: {mf1:.3f}')
    print(classification_report(yt,yp,target_names=labels,zero_division=0))
    cm=confusion_matrix(yt,yp,labels=range(len(labels)))
    fig,ax=plt.subplots(figsize=(7,6))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=labels,yticklabels=labels,ax=ax)
    ax.set_xlabel('Predicted');ax.set_ylabel('True')
    ax.set_title(f'{task.title()} — {name.upper()}')
    plt.xticks(rotation=30,ha='right');plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR/f'confusion_{task}_{name}.png',dpi=150);plt.show()
    return {'mean_acc':ma,'std':sa,'f1':mf1}

cph_s=show_results('cnn',cnn_pht,cnn_php,cnn_fpa,cfg.PHASE_LABELS,'phase')
lph_s=show_results('lstm',lstm_pht,lstm_php,lstm_fpa,cfg.PHASE_LABELS,'phase')
cpr_s=show_results('cnn',cnn_prt,cnn_prp,cnn_fra,cfg.PRESSURE_LABELS,'pressure')
lpr_s=show_results('lstm',lstm_prt,lstm_prp,lstm_fra,cfg.PRESSURE_LABELS,'pressure')

## 11 — Model comparison

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,task,cs,ls in [(axes[0],'Phase',cph_s,lph_s),(axes[1],'Pressure',cpr_s,lpr_s)]:
    x=np.arange(2);w=0.35
    ax.bar(x-w/2,[cs['mean_acc'],ls['mean_acc']],w,yerr=[cs['std'],ls['std']],
           label='Accuracy',capsize=5,color='#4C72B0')
    ax.bar(x+w/2,[cs['f1'],ls['f1']],w,label='Macro F1',color='#DD8452')
    ax.set_xticks(x);ax.set_xticklabels(['CNN','LSTM'])
    ax.set_ylim(0,1);ax.set_title(f'{task} Task');ax.legend()
plt.suptitle(f'{cfg.K_FOLDS}-Fold CV — Model Comparison',fontsize=14)
plt.tight_layout();plt.savefig(cfg.OUTPUT_DIR/'model_comparison.png',dpi=150);plt.show()

print(f'\nFINAL COMPARISON')
for n,ps,prs in [('CNN',cph_s,cpr_s),('LSTM',lph_s,lpr_s)]:
    print(f'  {n:5s} phase={ps["mean_acc"]:.3f}±{ps["std"]:.3f} (F1={ps["f1"]:.3f}) '
          f'pressure={prs["mean_acc"]:.3f}±{prs["std"]:.3f} (F1={prs["f1"]:.3f})')

## 12 — Save to Drive

In [ ]:
import shutil
drive_out = Path('/content/drive/MyDrive/mma_outputs')
drive_out.mkdir(parents=True, exist_ok=True)
for f in cfg.OUTPUT_DIR.iterdir():
    shutil.copy2(f, drive_out / f.name); print(f'  {f.name}')
print(f'\nSaved to {drive_out}')